In [1]:
# Imports

import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("data/structured_routing.csv")
dd = pd.read_csv("data/data_dictionary.csv")

# Take the column roles from the dictionary
FEATURES = dd.loc[dd.role == "pre_routing_feature", "column"].tolist()
TARGETS = dd.loc[dd.role == "target", "column"].tolist()
EVAL_ONLY = dd.loc[dd.role == "evaluation_only", "column"].tolist()
IDS = dd.loc[dd.role == "identifier", "column"].tolist()

print(len(FEATURES), len(TARGETS), len(EVAL_ONLY), len(IDS))
assert len(FEATURES) == 19 # Make sure the number of features is correct
assert set(FEATURES).isdisjoint(TARGETS + EVAL_ONLY + IDS) # Make sure the features are disjoint from the other roles

TARGET = "brier_improvement" # Primary target, as per the instructions (if we decide to include a cost prediction model as well, there will be a separate target for that)

19 3 23 4


In [5]:
train = df[df.split == "train"].copy()
val = df[df.split == "validation"].copy()
test = df[df.split == "test"].copy()

print(len(train), len(val), len(test))
print(train.question_id.nunique(), val.question_id.nunique(), test.question_id.nunique())

# 1) No question appears in two splits
q_tr, q_va, q_te = map(lambda d: set(d.question_id), (train, val, test))
assert q_tr.isdisjoint(q_va) and q_tr.isdisjoint(q_te) and q_va.isdisjoint(q_te)

# 2) Every train date is earlier than every val date, which is earlier than every test date
assert train.date.max() < val.date.min() and val.date.max() < test.date.min()

# 3) The target and evaluation-only columns are ALL missing in the test rows.
assert test[TARGETS + EVAL_ONLY].isna().all().all()

# 4) The target columns are fully present (no NaN) in train and val, so we have labels for training and for validation scoring.
assert train[TARGETS].notna().all().all() and val[TARGETS].notna().all().all()

714 170 215
151 34 43


In [7]:
# Feature sets for ablations

UNCERTAINTY = ["direct_uncertainty"]

MARKET = [
    "external_market_probability",
    "external_market_uncertainty",
    "direct_market_disagreement",
]

USAGE = [
    "direct_prompt_tokens",
    "direct_completion_tokens",
    "direct_reasoning_tokens",
    "direct_tool_calls",
    "direct_llm_calls",
    "direct_agent_cost",
    "direct_filter_cost",
    "direct_total_cost",
]

FEATURE_SETS = {
    "all": FEATURES,
    "uncertainty": UNCERTAINTY,
    "market": MARKET,
    "usage": USAGE,
    "combined": UNCERTAINTY + MARKET + USAGE,
}

In [ ]:
# Inputs and labels (Inputs are the same for both the benefit and the cost models, the targets differ)
CATEGORICAL_COLUMNS = ["source", "yhat_direct"]

# Only training data determines which categories get their own columns.
categories = {
    column: pd.CategoricalDtype(categories=sorted(train[column].unique()))
    for column in CATEGORICAL_COLUMNS
}

def make_features(split, columns):
    X = split[columns].copy()

    for column in X.select_dtypes(include="bool").columns:
        X[column] = X[column].astype(int)

    categorical = [column for column in CATEGORICAL_COLUMNS if column in X]
    for column in categorical:
        X[column] = X[column].astype(categories[column])

    return pd.get_dummies(X, columns=categorical, dtype=int)

X_train = {name: make_features(train, cols) for name, cols in FEATURE_SETS.items()}
X_val = {name: make_features(val, cols) for name, cols in FEATURE_SETS.items()}
X_test = {name: make_features(test, cols) for name, cols in FEATURE_SETS.items()}

y_train_benefit = train[TARGET].to_numpy()
y_val_benefit = val[TARGET].to_numpy()

y_train_cost = train["routed_agent_cost"].to_numpy()
y_val_cost = val["routed_agent_cost"].to_numpy()

for split in (train, val, test):
    assert split[FEATURES].notna().all().all()

assert train["routed_agent_cost"].notna().all()
assert val["routed_agent_cost"].notna().all()

# Keep all rows for a question in the same cross-validation fold
groups_train = train["question_id"].to_numpy()